In [21]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE

load_dotenv(override=True)

True

In [13]:
model = "gemini-3.5-flash-lite"
llm = ChatGoogleGenerativeAI(
    model=model,
    google_api_key=os.getenv("GEMINI_API_KEY")
)

# response = llm.invoke("Hello")
# print(response.content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [14]:
db_name = "vector_db"
knowledge_base_path = r"C:\training projects\RAG\knowledge_base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"found {len(files)} files")

entire_knowledge_base=""
for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base = f.read()
        entire_knowledge_base+="\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base)}")

found 27 files
Total characters in knowledge base: 220


In [15]:
token_count = llm.get_num_tokens(entire_knowledge_base)
print(token_count)

62


In [16]:
folders = glob.glob(r"C:\training projects\RAG\knowledge_base\*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 27 documents


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")


embeddings = GoogleGenerativeAIEmbeddings(
model="models/gemini-embedding-2-preview",
google_api_key=os.getenv("GEMINI_API_KEY")
)

#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Divided into 27 chunks
First chunk:

page_content='# about

NovaMind Analytics is a fictitious AI company founded in 2021, headquartered in San Francisco with regional offices in London, Singapore, and Bengaluru. It builds enterprise AI platforms for insurance, retail, and financial services, and has raised a fictitious Series C of USD 85 million led by Meridian Ventures.' metadata={'source': 'C:\\training projects\\RAG\\knowledge_base\\company\\about.md', 'doc_type': 'company'}
Vectorstore created with 27 documents
There are 27 vectors with 3,072 dimensions in the vector store
